# Chapter 2: Modern Optimization & Regularization (Adam, Dropout, & Weight Decay)

Welcome to **Chapter 2**!

In Chapter 1, we learned the fundamentals of an artificial neuron and trained a 2-layer network.
In this chapter, we level up to deeper networks (3+ layers) and tackle the two biggest challenges in modern deep learning:
1. **The Optimization Challenge**: How to train fast and reliably using the **Adam (Adaptive Moment Estimation)** optimizer.
2. **The Overfitting Challenge**: How to stop deep networks from memorizing noise using **Inverted Dropout** and **L2 Weight Decay**.

---

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.dataset import load_mnist, get_batches
from src.deep_model import DeepNeuralNetwork
from src.optimizers import SGDWithMomentum, Adam

print("Chapter 2 modules loaded from repo root:", repo_root)

## 1. Load the MNIST Dataset

We load our normalized MNIST dataset ($50,000$ train, $10,000$ validation, $10,000$ test):

In [ ]:
data_path = repo_root / "data" / "mnist"
X_train, y_train, X_val, y_val, X_test, y_test = load_mnist(
    data_dir=str(data_path),
    val_size=10000,
    flatten=True,
    normalize=True,
)

print(f"Training set   : {X_train.shape[0]:,} images")
print(f"Validation set : {X_val.shape[0]:,} images")
print(f"Test set       : {X_test.shape[0]:,} images")

## 2. Visualizing Inverted Dropout in Action

During training, **Dropout** randomly sets a fraction $p$ (e.g. 20%) of neuron activations to zero, scaling the survivors by $\frac{1}{1 - p}$.
Let us inspect what a hidden activation vector looks like with and without dropout:

In [ ]:
toy_net = DeepNeuralNetwork(layer_dims=[784, 64, 10], dropout_rate=0.25, seed=42)
sample_x = X_train[:1]

# 1. Forward in training mode (dropout active)
toy_net.train_mode()
toy_net.forward(sample_x)
train_activations = toy_net.cache["A1"][0]

# 2. Forward in evaluation mode (dropout inactive)
toy_net.eval_mode()
toy_net.forward(sample_x)
eval_activations = toy_net.cache["A1"][0]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.bar(range(len(train_activations)), train_activations, color="#dc2626", alpha=0.8)
plt.title("Training Mode (Dropout p=0.25 Active)", fontsize=11, fontweight="bold")
plt.xlabel("Neuron Index")
plt.ylabel("Activation")

plt.subplot(1, 2, 2)
plt.bar(range(len(eval_activations)), eval_activations, color="#2563eb", alpha=0.8)
plt.title("Evaluation Mode (Dropout Inactive)", fontsize=11, fontweight="bold")
plt.xlabel("Neuron Index")
plt.ylabel("Activation")

plt.tight_layout()
plt.show()

zero_count = np.sum(train_activations == 0)
print(f"During training, {zero_count}/{len(train_activations)} neurons were silenced ({zero_count/len(train_activations)*100:.1f}%).")

## 3. Comparative Training: Model A (SGD) vs Model B (Adam + Regularization)

We train two deep 3-layer networks ($784 \to 256 \to 128 \to 10$):
- **Model A**: SGD with Momentum (No Dropout, No Weight Decay)
- **Model B**: Adam Optimizer + Inverted Dropout ($p=0.2$) + L2 Weight Decay ($\lambda = 10^{-4}$)

In [ ]:
def train_deep_model(model, optimizer, epochs=6, batch_size=64):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for ep in range(1, epochs + 1):
        model.train_mode()
        for X_b, y_b in get_batches(X_train, y_train, batch_size=batch_size, shuffle=True):
            model.forward(X_b)
            grads = model.backward(y_b)
            for p_name in model.params.keys():
                model.params[p_name] = optimizer.step(p_name, model.params[p_name], grads[p_name])
        
        tr_l, tr_a = model.evaluate(X_train[:10000], y_train[:10000])
        v_l, v_a = model.evaluate(X_val, y_val)
        history["train_loss"].append(tr_l)
        history["train_acc"].append(tr_a)
        history["val_loss"].append(v_l)
        history["val_acc"].append(v_a)
        print(f"  Epoch {ep}/{epochs} | Train Acc: {tr_a*100:.2f}% | Val Acc: {v_a*100:.2f}%")
    return history

layer_dims = [784, 256, 128, 10]

print("Training Model A (Deep MLP with SGD, No Regularization)...")
net_a = DeepNeuralNetwork(layer_dims=layer_dims, dropout_rate=0.0, weight_decay=0.0, seed=42)
hist_a = train_deep_model(net_a, SGDWithMomentum(lr=0.03, momentum=0.9), epochs=6)

print("\nTraining Model B (Deep MLP with Adam + Dropout + Weight Decay)...")
net_b = DeepNeuralNetwork(layer_dims=layer_dims, dropout_rate=0.2, weight_decay=1e-4, seed=42)
hist_b = train_deep_model(net_b, Adam(lr=0.001), epochs=6)

## 4. Visualizing Results: SGD vs Adam & The Generalization Gap

In [ ]:
epochs_range = range(1, 7)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Validation Accuracy
ax1.plot(epochs_range, [a * 100 for a in hist_a["val_acc"]], "o--", color="#dc2626", label="Model A (SGD) Val Acc")
ax1.plot(epochs_range, [b * 100 for b in hist_b["val_acc"]], "s-", color="#059669", linewidth=2, label="Model B (Adam+Dropout) Val Acc")
ax1.set_title("Validation Accuracy: SGD vs Adam", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy (%)")
ax1.grid(True, alpha=0.3)
ax1.legend()

# 2. Overfitting Gap
gap_a = [t - v for t, v in zip(hist_a["train_acc"], hist_a["val_acc"])]
gap_b = [t - v for t, v in zip(hist_b["train_acc"], hist_b["val_acc"])]

ax2.plot(epochs_range, [g * 100 for g in gap_a], "o--", color="#dc2626", label="Model A Gap (Train - Val)")
ax2.plot(epochs_range, [g * 100 for g in gap_b], "s-", color="#059669", linewidth=2, label="Model B Gap (With Dropout)")
ax2.set_title("Generalization Gap: Overfitting vs Regularization", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Train Acc - Val Acc (%)")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Final Test Set Evaluation

Let us evaluate both models on the strictly unseen 10,000 test images:

In [ ]:
loss_a, acc_a = net_a.evaluate(X_test, y_test)
loss_b, acc_b = net_b.evaluate(X_test, y_test)

print(f"Model A (Unregularized SGD) Test Accuracy : {acc_a * 100:.2f}%")
print(f"Model B (Adam + Dropout)     Test Accuracy : {acc_b * 100:.2f}%")